In [1]:
import os
import warnings

# Silenciar advertencias de Python y arranque de Spark/Hadoop
warnings.filterwarnings("ignore")
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.showConsoleProgress=false pyspark-shell"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ED_03_02_Fundamentos_RDD") \
    .config("spark.driver.extraJavaOptions", "-Dlog4j.rootCategory=ERROR,console") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(f"✅ Spark iniciado correctamente y limpio. Versión: {spark.version}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/23 00:24:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark iniciado correctamente y limpio. Versión: 3.5.3


In [ ]:
Ejercicio 1: Word Count Básico

In [2]:
# Ejercicio 1: Word Count Básico
texto = ["hola mundo", "hola spark", "mundo spark"]
rdd_texto = sc.parallelize(texto)

conteo_palabras = (
    rdd_texto
    .flatMap(lambda linea: linea.split(" "))
    .map(lambda palabra: (palabra, 1))
    .reduceByKey(lambda a, b: a + b)
)

print("Ejercicio 1 - Conteo de palabras:")
for palabra, total in conteo_palabras.collect():
    print(f"  {palabra}: {total}")

Ejercicio 1 - Conteo de palabras:
  mundo: 2
  hola: 2
  spark: 2


In [ ]:
Ejercicio 2: Análisis de Ventas

In [3]:
# Ejercicio 2: Análisis de Ventas
ventas = [("Madrid", 100), ("Barcelona", 200), ("Madrid", 50)]
rdd_ventas = sc.parallelize(ventas)

resultado_ventas = (
    rdd_ventas
    .reduceByKey(lambda a, b: a + b)          # 1. Suma las ventas por ciudad
    .filter(lambda x: x[1] > 100)            # 2. Filtra ciudades con ventas > 100
    .sortBy(lambda x: x[1], ascending=False) # 3. Ordena de mayor a menor
)

print("Ejercicio 2 - Ciudades con ventas > 100 (mayor a menor):")
print(resultado_ventas.collect())

Ejercicio 2 - Ciudades con ventas > 100 (mayor a menor):
[('Barcelona', 200), ('Madrid', 150)]


In [ ]:
Ejercicio 3: Filtrado y Transformación

In [4]:
# Ejercicio 3: Filtrado y Transformación
precios_usd = [10, 25, 40, 5, 100, 15]
rdd_precios = sc.parallelize(precios_usd)

precios_eur = (
    rdd_precios
    .filter(lambda p: p >= 20)
    .map(lambda p: round(p * 0.9, 2))
)

print("Ejercicio 3 - Precios >= 20 USD convertidos a EUR (x0.9):")
print(precios_eur.collect())

Ejercicio 3 - Precios >= 20 USD convertidos a EUR (x0.9):
[22.5, 36.0, 90.0]


In [ ]:
Ejercicio 4: Procesamiento de CSV

In [5]:
# Ejercicio 4: Procesamiento de CSV
csv_datos = ["Leche,Huevos,Pan", "Manzanas,Peras", "Leche,Chocolate"]
rdd_csv = sc.parallelize(csv_datos)

productos_unicos = (
    rdd_csv
    .flatMap(lambda linea: linea.split(","))
    .distinct()
)

lista_unicos = productos_unicos.collect()
print("Ejercicio 4 - Productos únicos:", lista_unicos)
print("Total de productos únicos:", productos_unicos.count())

Ejercicio 4 - Productos únicos: ['Pan', 'Peras', 'Leche', 'Huevos', 'Manzanas', 'Chocolate']
Total de productos únicos: 6


In [ ]:
Ejercicio 5: Análisis de Logs

In [6]:
import os
import shutil

# 1. Crear el archivo simulado en la carpeta datos/
os.makedirs("datos", exist_ok=True)
ruta_logs = "datos/logs_simulados.txt"

with open(ruta_logs, "w") as f:
    f.write("192.168.1.1,10:00,GET,/index\n")
    f.write("192.168.1.2,10:01,POST,/login\n")
    f.write("192.168.1.1,10:02,GET,/imagen.jpg\n")

# 2. Cargar el archivo con Spark
rdd_logs = sc.textFile(ruta_logs)

# 3. Contar peticiones por IP y filtrar las que tienen > 1 petición
ips_frecuentes = (
    rdd_logs
    .map(lambda linea: (linea.split(",")[0], 1))
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda x: x[1] > 1)
)

print("Ejercicio 5 - IPs con más de 1 petición:", ips_frecuentes.collect())

# 4. Guardar el resultado en un archivo de texto (limpiando si ya existía)
ruta_salida = "datos/salida_ips"
if os.path.exists(ruta_salida):
    shutil.rmtree(ruta_salida)

ips_frecuentes.saveAsTextFile(ruta_salida)
print(f"✅ Resultado guardado en: {ruta_salida}")

Ejercicio 5 - IPs con más de 1 petición: [('192.168.1.1', 2)]
✅ Resultado guardado en: datos/salida_ips


In [ ]:
Ejercicio 6: Pipeline y Shuffle

In [7]:
# Ejercicio 6: Pipeline y Shuffle
ventas_ej6 = [("Madrid", 100), ("Barcelona", 200), ("Madrid", 50), ("Valencia", 50)]
rdd_ventas6 = sc.parallelize(ventas_ej6)

pipeline_ventas = (
    rdd_ventas6
    .filter(lambda x: x[1] >= 60)              # 1. Filtra las ventas menores de 60
    .reduceByKey(lambda a, b: a + b)          # 2. Suma las ventas por ciudad
    .sortBy(lambda x: x[1], ascending=False)  # 3. Ordena por facturación (mayor a menor)
)

print("Ejercicio 6 - Facturación ordenada (ventas >= 60):")
print(pipeline_ventas.collect())

Ejercicio 6 - Facturación ordenada (ventas >= 60):
[('Barcelona', 200), ('Madrid', 100)]


In [ ]:
Ejercicio 7: Diferencia Map vs FlatMap

In [8]:
# Ejercicio 7: Diferencia Map vs FlatMap
frases = ["Hola mundo", "Apache Spark es guay"]
rdd_frases = sc.parallelize(frases)

# 1. Aplicando map()
res_map = rdd_frases.map(lambda f: f.split(" ")).collect()

# 2. Aplicando flatMap()
res_flatmap = rdd_frases.flatMap(lambda f: f.split(" ")).collect()

print("1. Resultado con map():", res_map)
print("   -> Estructura: Lista anidada (lista de listas), mantiene 1 elemento de salida por cada frase.")
print("\n2. Resultado con flatMap():", res_flatmap)
print("   -> Estructura: Lista plana única, 'aplana' todas las palabras en un solo nivel.")

1. Resultado con map(): [['Hola', 'mundo'], ['Apache', 'Spark', 'es', 'guay']]
   -> Estructura: Lista anidada (lista de listas), mantiene 1 elemento de salida por cada frase.

2. Resultado con flatMap(): ['Hola', 'mundo', 'Apache', 'Spark', 'es', 'guay']
   -> Estructura: Lista plana única, 'aplana' todas las palabras en un solo nivel.


In [ ]:
Ejercicio 8: Optimización

In [9]:
# Ejercicio 8: Optimización de un RDD reutilizado 5 veces
print("Ejercicio 8 - Respuesta de optimización:")
print("""
Si un RDD se utiliza 5 veces en diferentes cálculos, debemos persistirlo en memoria 
usando .cache() o .persist(StorageLevel.MEMORY_AND_DISK) antes de la primera acción.

¿Por qué?
Por defecto, Spark evalúa de forma perezosa (Lazy Evaluation) y recalcula todo el linaje 
(DAG) desde cero cada vez que se ejecuta una acción. Al cachearlo, Spark calcula el RDD 
una sola vez, lo guarda en RAM (o RAM + disco) y las 4 ejecuciones restantes leen 
directamente de caché en milisegundos. Al terminar los 5 cálculos, liberamos la memoria 
con .unpersist().
""")

Ejercicio 8 - Respuesta de optimización:

Si un RDD se utiliza 5 veces en diferentes cálculos, debemos persistirlo en memoria 
usando .cache() o .persist(StorageLevel.MEMORY_AND_DISK) antes de la primera acción.

¿Por qué?
Por defecto, Spark evalúa de forma perezosa (Lazy Evaluation) y recalcula todo el linaje 
(DAG) desde cero cada vez que se ejecuta una acción. Al cachearlo, Spark calcula el RDD 
una sola vez, lo guarda en RAM (o RAM + disco) y las 4 ejecuciones restantes leen 
directamente de caché en milisegundos. Al terminar los 5 cálculos, liberamos la memoria 
con .unpersist().

